<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Camera Calibration</b></h1>
</div>

## Theoretical Foundations

The theory is organized in the same **13-task order** as the implementation notebook. Sections involving file handling or visualization state the mathematical role of that task; sections involving calibration derive the exact equations used in code.

## 1. Load the Sorted JPEG Calibration Images

This task is organizational rather than mathematical. Deterministic filename ordering ensures that the same set of calibration views is processed in the same order on every execution.

## 2. Detect and Refine Chessboard Corners

Each valid image provides measured pixel coordinates

$$
\mathbf{x}_i=[u_i,v_i]^T.
$$

Sub-pixel refinement improves the precision of these observations before homography estimation. The retained pattern contains $8\times6=48$ internal corners.

## 3. Build the Planar World Coordinates

The chessboard is planar, so every calibration point satisfies $Z=0$. The implementation uses

$$
\mathbf{X}_{p,i}=[X_i,Y_i,1]^T
$$

with $0.03\,\mathrm{m}$ spacing, the first corner at $(0,0)$, $X$ along the 8-corner direction and $Y$ along the 6-corner direction.

## 4. Compute $T_{\mathrm{image}}$ and $T_{\mathrm{plane}}$

For either 2D point set, let $(\bar x,\bar y)$ be the centroid and $\bar d$ the mean Euclidean distance from that centroid. The scale is

$$
s_n=\frac{\sqrt{2}}{\bar d},
$$

and

$$
T=
\begin{bmatrix}
s_n&0&-s_n\bar x\\
0&s_n&-s_n\bar y\\
0&0&1
\end{bmatrix}.
$$

Separate transforms are computed for image and planar points.

## 5. Build the DLT Matrix $Q$ and Solve $Q\mathbf{h}=0$ by SVD

For each normalized correspondence $(X,Y)\leftrightarrow(u,v)$, DLT contributes

$$
\begin{bmatrix}
X&Y&1&0&0&0&-uX&-uY&-u
\end{bmatrix},
$$

$$
\begin{bmatrix}
0&0&0&X&Y&1&-vX&-vY&-v
\end{bmatrix}.
$$

All rows form $Q$, and

$$
Q\mathbf{h}=0.
$$

If

$$
Q=U\Sigma V^T,
$$

the last right singular vector gives $\mathbf{h}$, which is reshaped into $H_n$.

## 6. Denormalize Each Homography

The normalized homography is mapped back to the original coordinate systems using

$$
H=T_{\mathrm{image}}^{-1}H_nT_{\mathrm{plane}}.
$$

Since a homography is defined only up to non-zero scale, the implementation fixes the scale using

$$
H\leftarrow\frac{H}{H_{33}}.
$$

## 7. Build the Zhang Matrix $V$ and Solve $Vb=0$ by SVD

Write

$$
H=[\mathbf{h}_1\;\mathbf{h}_2\;\mathbf{h}_3],
\qquad
B=K^{-T}K^{-1}.
$$

The first two rotation columns imply

$$
\mathbf{h}_1^TB\mathbf{h}_2=0,
$$

$$
\mathbf{h}_1^TB\mathbf{h}_1-\mathbf{h}_2^TB\mathbf{h}_2=0.
$$

Using

$$
v_{ij}=
\begin{bmatrix}
h_{i1}h_{j1}\\
h_{i1}h_{j2}+h_{i2}h_{j1}\\
h_{i2}h_{j2}\\
h_{i3}h_{j1}+h_{i1}h_{j3}\\
h_{i3}h_{j2}+h_{i2}h_{j3}\\
h_{i3}h_{j3}
\end{bmatrix},
$$

each view contributes $v_{12}$ and $v_{11}-v_{22}$. Stacking all views gives

$$
Vb=0,
$$

which is solved by SVD.

## 8. Recover $\alpha,\beta,\gamma,u_0,v_0$ and Construct $K$

Let

$$
b=[b_{11},b_{12},b_{22},b_{13},b_{23},b_{33}]^T
$$

and

$$
d=b_{11}b_{22}-b_{12}^2.
$$

Then

$$
v_0=\frac{b_{12}b_{13}-b_{11}b_{23}}{d},
$$

$$
\lambda=
b_{33}
-
\frac{b_{13}^2+v_0(b_{12}b_{13}-b_{11}b_{23})}{b_{11}},
$$

$$
\alpha=\sqrt{\frac{\lambda}{b_{11}}},
\qquad
\beta=\sqrt{\frac{\lambda b_{11}}{d}},
$$

$$
\gamma=-\frac{b_{12}\alpha^2\beta}{\lambda},
$$

$$
u_0=
\frac{\gamma v_0}{\beta}
-
\frac{b_{13}\alpha^2}{\lambda}.
$$

Finally,

$$
K=
\begin{bmatrix}
\alpha&\gamma&u_0\\
0&\beta&v_0\\
0&0&1
\end{bmatrix}.
$$

## 9. Recover $R$ and $t$ for Every Retained View

For

$$
H=[\mathbf{h}_1\;\mathbf{h}_2\;\mathbf{h}_3],
$$

the pose scale is

$$
\lambda_p=\frac{1}{\lVert K^{-1}\mathbf{h}_1\rVert}.
$$

Then

$$
\mathbf{r}_1=\lambda_pK^{-1}\mathbf{h}_1,
\qquad
\mathbf{r}_2=\lambda_pK^{-1}\mathbf{h}_2,
$$

$$
\mathbf{r}_3=\mathbf{r}_1\times\mathbf{r}_2,
\qquad
t=\lambda_pK^{-1}\mathbf{h}_3.
$$

The approximate rotation is projected to the nearest proper rotation matrix by SVD, enforcing $\det(R)=+1$.

## 10. Reproject the $Z=0$ Calibration Points

Each planar point is embedded in 3D as

$$
\mathbf{X}_{3D}=[X,Y,0]^T.
$$

The camera-frame coordinate is

$$
\mathbf{X}_c=R\mathbf{X}_{3D}+t,
$$

and the homogeneous image point is

$$
\tilde{\mathbf{x}}=K\mathbf{X}_c.
$$

Pixel coordinates are obtained by dividing the first two components by the third.

## 11. Compute Point-wise Errors, Mean Error and RMSE

For measured point $\mathbf{x}_i$ and prediction $\hat{\mathbf{x}}_i$,

$$
e_i=\lVert\hat{\mathbf{x}}_i-\mathbf{x}_i\rVert_2.
$$

For $N$ points,

$$
\bar e=\frac{1}{N}\sum_{i=1}^{N}e_i,
$$

$$
\mathrm{RMSE}=\sqrt{\frac{1}{N}\sum_{i=1}^{N}e_i^2}.
$$

The same definitions are used per view and globally.

## 12. Produce and Save the Six Required Diagnostic Figures

The six figures visualize complementary parts of the mathematical solution:

### 12.1 Homography Estimation Pipeline
Shows the sequence from correspondences to normalized DLT and $H$.

### 12.2 Mean Reprojection Error by View
Compares $\bar e$ across calibration views.

### 12.3 Reprojection Error Distribution
Shows the empirical distribution of $e_i$.

### 12.4 Detected Chessboard Corners
Verifies the measured observations $\mathbf{x}_i$.

### 12.5 Estimated Camera Poses
Uses the camera centre

$$
C=-R^Tt.
$$

### 12.6 Detected vs Reprojected Points
Compares $\mathbf{x}_i$ with $\hat{\mathbf{x}}_i$.

## 13. Run the Numerical and Output-file Validation Checks

The final mathematical validity conditions are:

$$
K\in\mathbb{R}^{3\times3}
$$

with finite entries,

$$
R^TR\approx I,
$$

$$
\det(R)\approx1,
$$

and finite reprojection errors $e_i$.

The implementation also verifies that all six required output files exist.

## Scope and Limitations

The complete 13-task solution uses a pinhole camera model only. Radial/tangential distortion parameters and nonlinear bundle-adjustment refinement are intentionally excluded, so the reported residuals measure the fit of this exact implemented model.